In [ ]:
!pip install tensorflow

In [2]:
import tensorflow as tf
print(tf.__version__)

2.19.0


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [5]:
dataset=pd.read_csv('Churn_Modelling.csv')
dataset.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [6]:
X=dataset.iloc[:,3:13]
Y=dataset.iloc[ : ,13]

In [7]:
# feature engineering

In [8]:
geography=pd.get_dummies(X['Geography'],drop_first=True).astype(int)
gender=pd.get_dummies(X['Gender'],drop_first=True).astype(int)

In [9]:
X=X.drop(['Geography','Gender'],axis=1)

In [10]:
X=pd.concat([X,gender,geography],axis=1)

In [11]:
X.head()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Male,Germany,Spain
0,619,42,2,0.00,1,1,1,101348.88,0,0,0
1,608,41,1,83807.86,1,0,1,112542.58,0,0,1
2,502,42,8,159660.80,3,1,0,113931.57,0,0,0
3,699,39,1,0.00,2,0,0,93826.63,0,0,0
4,850,43,2,125510.82,1,1,1,79084.10,0,0,1


In [12]:
#train test split
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(X,Y,test_size=0.2,random_state=0)

In [13]:
#scaling is required
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()
x_train=sc.fit_transform(x_train)
x_test=sc.transform(x_test)

In [14]:
x_train.shape


(8000, 11)

In [15]:
## creating ANN

In [19]:
from tensorflow.keras.models import Sequential  #used to do forward and backward passes
from tensorflow.keras.layers import Dense        #used to create input layer, hidden layer, output layer
from tensorflow.keras.layers import LeakyReLU,ReLU,PReLU,ELU  #activation functions
from tensorflow.keras.layers import Dropout     #to reduce overfitting

In [20]:
#initialize ANN
classifier=Sequential()

In [22]:
#create the input layer
classifier.add(Dense(units=11,activation='relu'))

In [24]:
#creating hidden layer 1
classifier.add(Dense(units=7,activation='relu'))

In [25]:
#creating hidden layer 2
classifier.add(Dense(units=6,activation='relu'))

In [26]:
#output layer
classifier.add(Dense(1,activation='sigmoid'))

In [33]:
#we will compile the ann
classifier.compile(optimizer=opt,loss='binary_crossentropy',metrics=['accuracy'])

In [32]:
#if you want to change the learning rate
import tensorflow
opt=tensorflow.keras.optimizers.Adam(learning_rate=0.01)

In [38]:
from tensorflow.keras.callbacks import EarlyStopping


In [41]:
early_stopping=EarlyStopping(
    monitor="val_loss",
    min_delta=0.0001,
    patience=20,
    verbose=1,
    mode="auto",
    baseline=None,
    restore_best_weights=False,
    start_from_epoch=0,
)

In [42]:
#train the ann
model_history=classifier.fit(x_train,y_train,validation_split=0.33,batch_size=10,epochs=1000,callbacks=early_stopping)

Epoch 1/1000
536/536 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8604 - loss: 0.3277 - val_accuracy: 0.8504 - val_loss: 0.3599
Epoch 2/1000
536/536 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8611 - loss: 0.3326 - val_accuracy: 0.8569 - val_loss: 0.3602
Epoch 3/1000
536/536 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.8659 - loss: 0.3257 - val_accuracy: 0.8573 - val_loss: 0.3634
Epoch 4/1000
536/536 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.8617 - loss: 0.3368 - val_accuracy: 0.8531 - val_loss: 0.3604
Epoch 5/1000
536/536 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8600 - loss: 0.3302 - val_accuracy: 0.8546 - val_loss: 0.3611
Epoch 6/1000
536/536 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8674 - loss: 0.3275 - val_accuracy: 0.8508 - val_loss: 0.3621
Epoch 7/1000
536/536 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.8665 - loss: 0.3293 - val_accuracy: 0.8470 - val_loss: 0.3752
Epoch 8/1000
536/536 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8650 - loss: 0.3249 - 

In [43]:
y_pred=classifier.predict(x_test)
y_pred=(y_pred>0.5)

63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step


In [46]:
from sklearn.metrics import confusion_matrix
cm=confusion_matrix(y_test,y_pred)
cm


array([[1511,   84],
       [ 195,  210]])

In [47]:
from sklearn.metrics import accuracy_score
acc=accuracy_score(y_test,y_pred)
acc

0.8605

In [48]:
model_history.history.keys()      #focused terms

dict_keys(['accuracy', 'loss', 'val_accuracy', 'val_loss'])

In [49]:
classifier.get_weights()

[array([[-0.46196198,  0.4060736 ,  1.1992557 ,  0.19295731, -0.3742874 ,
          0.45112127,  0.64182824, -0.16918091,  0.30328074,  0.77025247,
          0.64162   ],
        [ 1.6099463 , -3.1801426 , -0.90894556, -1.3443316 , -0.48597354,
         -3.2527864 , -0.16513452,  2.5669951 ,  1.2001034 , -1.091983  ,
          0.5239059 ],
        [-0.5898369 , -0.00907198, -0.10099142,  1.9744612 , -0.33291698,
         -0.20054433, -0.01676853, -0.41702503, -0.31029096,  1.6693649 ,
          0.16829453],
        [-2.0606306 ,  1.7961296 ,  1.652641  , -0.35453126, -1.2165575 ,
          0.49823087, -2.4077625 ,  0.41175795,  1.6971633 , -2.1888404 ,
          1.927496  ],
        [-3.6632571 ,  1.421492  , -0.9335873 ,  1.7364247 , -1.3252878 ,
          0.19507642,  2.5825574 ,  0.11905346,  3.2839112 , -0.57722944,
         -1.1839088 ],
        [-0.23214364,  0.02531472, -0.7738083 , -0.5990947 ,  1.5890439 ,
         -0.39642176,  0.22858849, -0.01193575, -0.42053086, -1.498499 